# T0 — probabilistic time-series forecasting

T0 is a time-series foundation model from The Forecasting Company. Give it a series'
recent history and it returns **quantile forecasts** — a range, not just a point
estimate — with no per-series training.

This notebook runs end to end with no edits beyond your Model Package ARN, and no
data for you to find: it generates a demo series, forecasts it two ways, and cleans
up after itself.

1. [Deploy a real-time endpoint](#realtime) — interactive, one series per call
2. [Condition on known-future covariates](#covariates) — calendar effects, promotions
3. [Run a batch transform job](#batch) — many series from a file, no server left running
4. [Clean up](#cleanup)

## Prerequisites

- **Subscribe to T0** in AWS Marketplace. Your subscription gives you a Model Package
  ARN specific to your AWS Region — paste it below.
- Run this in a SageMaker environment (SageMaker Studio is easiest) with an execution
  role that can create models, endpoints and transform jobs, and read/write S3.
- Deploy in the **same Region** as your subscription.

The container runs **network-isolated** — no internet access, in or out. The weights
are baked into the image, so your data never leaves your AWS account.

### Install the SDK this notebook targets

The SageMaker Python SDK v3 replaced the high-level classes used below, so pin v2.
You will see a deprecation notice when it loads — that is expected, and v2 remains
the documented path for deploying a subscribed model package.

In [ ]:
%pip install --quiet "sagemaker>=2.200,<3"

In [ ]:
MODEL_PACKAGE_ARN = "<YOUR_SUBSCRIPTION_MODEL_PACKAGE_ARN>"

# T0 runs on GPU for both real-time endpoints and batch transform jobs.
REALTIME_INSTANCE_TYPE = "ml.g4dn.xlarge"
TRANSFORM_INSTANCE_TYPE = "ml.g4dn.xlarge"

import json
import uuid

import numpy as np
import sagemaker
from sagemaker import ModelPackage, get_execution_role
from sagemaker.deserializers import JSONDeserializer
from sagemaker.predictor import Predictor
from sagemaker.s3 import S3Downloader, S3Uploader
from sagemaker.serializers import JSONSerializer

session = sagemaker.Session()
role = get_execution_role()
bucket = session.default_bucket()
run_id = uuid.uuid4().hex[:8]

print(f"region  {session.boto_region_name}")
print(f"bucket  {bucket}")

## The demo series

Hourly demand for a single store over ten weeks: a daily shape, a weaker weekly
shape, a gentle upward trend, and noise. Two short gaps are left as `NaN` — T0
treats those as missing rather than zero, so you do not have to interpolate first.

We hold back the final 48 hours to compare against.

In [ ]:
rng = np.random.default_rng(0)

HOURS = 24 * 7 * 10
HORIZON = 48

t = np.arange(HOURS)
daily = 12.0 * np.sin(2 * np.pi * (t % 24) / 24 - 1.2)
weekly = 4.0 * np.sin(2 * np.pi * (t % 168) / 168)
trend = 0.004 * t
series = 60.0 + daily + weekly + trend + rng.normal(0, 2.0, HOURS)

# Two outages: recorded as missing, not as zero demand.
series[300:308] = np.nan
series[900:904] = np.nan

history, holdout = series[:-HORIZON], series[-HORIZON:]
print(f"{len(history)} hours of history, {int(np.isnan(history).sum())} missing")
print(f"holding back {len(holdout)} hours to compare against")

<a id="realtime"></a>
## 1. Deploy a real-time endpoint

Deploys straight from the Model Package ARN. Expect a few minutes while the instance
comes up.

In [ ]:
model = ModelPackage(
    role=role,
    model_package_arn=MODEL_PACKAGE_ARN,
    sagemaker_session=session,
    # Without predictor_cls, deploy() returns None instead of a predictor.
    predictor_cls=Predictor,
)
predictor = model.deploy(
    initial_instance_count=1,
    instance_type=REALTIME_INSTANCE_TYPE,
    endpoint_name=f"t0-demo-{run_id}",
    serializer=JSONSerializer(),
    deserializer=JSONDeserializer(),
)
print("endpoint ready:", predictor.endpoint_name)

### Forecast

Three fields. `context` is the history, `horizon` is how many steps ahead, and
`quantiles` are the levels you want back — ascending, strictly between 0 and 1.

The response gives `quantiles` shaped `[horizon][n_quantiles]` and echoes
`quantile_levels` in the order you asked for.

**Sending gaps:** use `NaN`, not `null`. JSON has no NaN literal, so Python's
`json.dumps` — which the SageMaker serializer uses — emits the non-standard `NaN`
token, and the model accepts it. A literal `null` is rejected with HTTP 422. If you
build payloads with a strict encoder such as `orjson`, it will refuse to emit `NaN`;
use the standard library's `json` for series that contain gaps.

In [ ]:
request = {
    # np.nan passes straight through: json.dumps writes it as the NaN token.
    "context": [float(v) for v in history],
    "horizon": HORIZON,
    "quantile_levels": [0.1, 0.5, 0.9],
}
response = predictor.predict(request)

levels = response["quantile_levels"]
forecast = np.array(response["quantiles"])
print(f"levels {levels}  forecast shape {forecast.shape}")
print("first 3 hours:", np.round(forecast[:3], 2).tolist())

### Read the interval, not just the line

The p10–p90 band is the useful output: it widens with genuine uncertainty. A point
forecast hides that.

In [ ]:
import matplotlib.pyplot as plt

lo, med, hi = forecast[:, 0], forecast[:, 1], forecast[:, 2]
past = np.arange(-24 * 7, 0)
future = np.arange(HORIZON)

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(past, history[-24 * 7:], color="#5B6B7B", lw=1, label="history")
ax.plot(future, holdout, color="#0E1621", lw=1.2, ls="--", label="actual (held back)")
ax.fill_between(future, lo, hi, color="#A96F26", alpha=0.22, label=f"p{levels[0]:.0%}-p{levels[-1]:.0%}")
ax.plot(future, med, color="#A96F26", lw=1.8, label="median")
ax.axvline(0, color="#9FAEBC", lw=0.8)
ax.set_xlabel("hours from now")
ax.set_ylabel("demand")
ax.legend(loc="upper left", frameon=False, ncols=4, fontsize=9)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

covered = np.mean((holdout >= lo) & (holdout <= hi))
print(f"{covered:.0%} of held-back hours fell inside the p10-p90 band")

<a id="covariates"></a>
## 2. Condition on known-future signals

If you know something about the future — a public holiday, a planned promotion, a
scheduled closure — pass it as `future_covariates` and T0 conditions on it.

Shape is `[n_covariates][len(context) + horizon]`: each covariate must be supplied
across the history *and* the horizon, which is what makes it "known-future". Values
are yours to define; here a single binary flag marks Sundays.

How much difference this makes depends on whether the signal actually carries
information in your history. The point of this cell is the API, not a guaranteed
improvement.

In [ ]:
hours = np.arange(len(history) + HORIZON)
is_sunday = ((hours // 24) % 7 == 6).astype(float)

with_cov = predictor.predict({
    **request,
    "future_covariates": [is_sunday.tolist()],
})

cov_forecast = np.array(with_cov["quantiles"])
shift = np.abs(cov_forecast[:, 1] - forecast[:, 1]).mean()
print(f"covariate shape: 1 x {len(is_sunday)} (context {len(history)} + horizon {HORIZON})")
print(f"mean absolute change in the median forecast: {shift:.3f}")

<a id="batch"></a>
## 3. Batch transform — many series, no server

For scheduled bulk scoring, a batch transform job is usually the better fit than an
endpoint: it starts instances, processes a file from S3, writes results back, and
shuts down. You pay for the job, not for uptime.

The input is **JSON Lines** — one forecast request per line. With `split_type="Line"`
and `strategy="SingleRecord"`, SageMaker sends each line to the model separately and
reassembles the responses in order, one per line.

In [ ]:
def demo_series(seed: int, hours: int, level: float) -> list:
    r = np.random.default_rng(seed)
    x = np.arange(hours)
    y = level + 10 * np.sin(2 * np.pi * (x % 24) / 24) + r.normal(0, 1.5, hours)
    return [float(v) for v in y]


batch_requests = [
    {"context": demo_series(1, 336, 40.0), "horizon": 24, "quantile_levels": [0.1, 0.5, 0.9]},
    {"context": demo_series(2, 504, 120.0), "horizon": 48, "quantile_levels": [0.1, 0.5, 0.9]},
    {"context": demo_series(3, 168, 8.0), "horizon": 12, "quantile_levels": [0.25, 0.5, 0.75]},
]

prefix = f"t0-demo/{run_id}"
input_uri = f"s3://{bucket}/{prefix}/input/series.jsonl"
S3Uploader.upload_string_as_file_body(
    "\n".join(json.dumps(r) for r in batch_requests) + "\n",
    input_uri,
    sagemaker_session=session,
)
print(f"{len(batch_requests)} series -> {input_uri}")

In [ ]:
transformer = ModelPackage(
    role=role,
    model_package_arn=MODEL_PACKAGE_ARN,
    sagemaker_session=session,
).transformer(
    instance_count=1,
    instance_type=TRANSFORM_INSTANCE_TYPE,
    strategy="SingleRecord",
    assemble_with="Line",
    accept="application/json",
    output_path=f"s3://{bucket}/{prefix}/output",
)
transformer.transform(data=input_uri, content_type="application/json", split_type="Line")
transformer.wait()

In [ ]:
body = S3Downloader.read_file(
    f"s3://{bucket}/{prefix}/output/series.jsonl.out",
    sagemaker_session=session,
)
results = [json.loads(line) for line in body.splitlines() if line.strip()]

for i, (req, res) in enumerate(zip(batch_requests, results), start=1):
    q = np.array(res["quantiles"])
    print(
        f"series {i}: {len(req['context'])} hours in -> {q.shape[0]} steps x "
        f"{q.shape[1]} quantiles, median starts at {q[0, q.shape[1] // 2]:.1f}"
    )

<a id="cleanup"></a>
## 4. Clean up

The transform job has already released its instances. The endpoint has not — delete
it, or it keeps billing.

In [ ]:
# Order matters: delete_model() finds the model by describing the endpoint config,
# and delete_endpoint() removes that config — so the model goes first.
predictor.delete_model()
predictor.delete_endpoint()

# The transform job registered a model of its own, which the predictor does not own.
session.sagemaker_client.delete_model(ModelName=transformer.model_name)
print("endpoint and models deleted")

## Reference

**Request**

| Field | Type | Required | Meaning |
| --- | --- | --- | --- |
| `context` | `list[float]` | yes | History, oldest first. `NaN` marks missing — not `null`, which returns HTTP 422. Up to 16,384 points. |
| `horizon` | `int` | yes | Steps to forecast, 1 to 2,048. |
| `quantiles` | `list[float]` | no | Levels in (0, 1), ascending. Defaults to `[0.1, 0.5, 0.9]`. |
| `future_covariates` | `list[list[float]]` | no | Shaped `[n_covariates][len(context) + horizon]`. |

**Response**

| Field | Type | Meaning |
| --- | --- | --- |
| `quantiles` | `list[list[float]]` | `[horizon][n_quantiles]`. |
| `quantile_levels` | `list[float]` | The levels, in the order requested. |

The endpoint also accepts a JSON **list** of requests and returns a list of
responses, which is what makes the batch path above work.

**Limits** — real-time payload ≤ 25 MB and ≤ 60 s per invocation; batch transform
≤ 100 MB per record and ≤ 60 min per job.